# Spatial Transcriptomics Analysis Workflow: ATERA WTA FFPE breast cancer example

**Authors and Contributors:** Sara Jimenez, Lukas Heumos, Severin Dicks

Tutorial notebook for the **ATERA WTA FFPE breast cancer** 10x image-based spatial transcriptomics dataset (170,057 cells × 18,028 genes, whole-transcriptome add-on panel), loaded as a `SpatialData` zarr store from the scverse example data.

**Analysis outline:**

0. Load the data 
1. Preprocessing  
   1.1 QC  
   1.2 normalization
2. Build the spatial graph
3. Cell type annotation
4. Spatial structure —  
   4.1 spatially aware clustering (niches)  
   4.2 niche characterization  
   4.3 co-occurrence / neighborhood enrichment / ligand-receptor signaling
6. Spatially variable genes & gene-level tasks


> **Interactive tutorial for the `process` capsule. It is not what the App Panel runs.**
> From rapids-singlecell's tutorials (`scverse/rapids-singlecell-tutorials` at `0b059a3`, `08_atera_demo.ipynb`), outputs stripped.
> Rendered original: <https://rapids-singlecell.scverse.org/en/latest/notebooks/08_atera_demo.html>
>
> Changes from the original:
> - Reads its input from a data asset (set `DATA_PATH` below) instead of downloading and unzipping it.
> - Needs internet for the OmniPath (PROGENy) and ligand-receptor lookups.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import time

import numpy as np
import cupy as cp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc
import squidpy as sq 
import rapids_singlecell as rsc
import spatialdata as sd

import omnipath as op

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 0

## 0. Load the dataset

### Input

A **SpatialData `.zarr` store on a data asset** (a folder, not the `.zarr.zip`), read with `spatialdata.read_zarr`:
- a table named **`table`**: AnnData with **raw counts in `X`** (sparse), cell centroids in **`obsm["spatial"]`**,
  and `obs` columns **`cell_id`** (matches the shapes) and **`nucleus_count`** (used to drop anucleate cells);
- a shapes element named **`cell_boundaries`**: the cell polygons the table is pointed at for plotting.

The tutorial used 10x's ATERA WTA FFPE breast cancer set (170,057 cells × 18,028 genes),
`atera_breast_cancer.zarr.zip` from `exampledata.scverse.org/rapids-singlecell`, unzipped. The cell-type markers
in section 3 are for breast cancer; other tissues need their own.


In [ ]:
notebook_start = time.time()

In [ ]:
# A SpatialData .zarr folder with a `table` and `cell_boundaries` shapes (see Input above).
DATA_PATH = Path("/data/<data asset mount>/atera_breast_cancer.zarr")

sdata = sd.read_zarr(DATA_PATH)
sdata


### Prepare AnnData from SpatialData object

spatialdata-plot needs the table to annotate the shapes element it will be rendered against. 
- By default the table's region metadata points at the label image we didn't load (`cells_labels`), so we repoint it at the cell-boundary polygons we did load (`cell_boundaries`).
- Move the AnnData object to GPU.

In [ ]:
table = sdata.tables["table"]
table.obs["region"] = pd.Categorical(["cell_boundaries"] * table.n_obs)
sdata.set_table_annotates_spatialelement(
    "table", region="cell_boundaries", region_key="region", instance_key="cell_id"
)

adata = sdata.tables["table"].copy()
adata.var_names_make_unique()
adata.layers["counts"] = adata.X.copy()  # keep raw counts around explicitly

rsc.get.anndata_to_GPU(adata, convert_all=True)
adata

## 1. Preprocessing

### 1.1 Quality control

We compute standard QC metrics with `rsc.pp.calculate_qc_metrics`
(GPU) and use percentile-based thresholds as an illustrative starting
point. 

In [ ]:
rsc.pp.calculate_qc_metrics(
    adata, 
    expr_type="counts", 
    var_type="genes", 
    log1p=True
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(adata.obs["total_counts"], bins=100, ax=axes[0])
axes[0].set_title("total_counts (pre-filter)")
sns.histplot(adata.obs["n_genes_by_counts"], bins=100, ax=axes[1])
axes[1].set_title("n_genes_by_counts (pre-filter)")
fig.tight_layout()

In [ ]:
# Drop failed segmentations: empty cells and anucleate fragments.

keep = (adata.obs["total_counts"] >= 10) & (adata.obs["nucleus_count"] >= 1)
adata = adata[keep].copy()
rsc.pp.filter_genes(adata, min_cells=3)

### 1.2 Normalization and dimensionality reduction 

- We use the standard scanpy-style size-factor normalization; used as the
single normalized layer (`lognorm`) for everything downstream.
- We choose 2000 HVG and 50 PCs. 

In [ ]:
rsc.pp.normalize_total(adata)
rsc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

In [ ]:
rsc.pp.highly_variable_genes(
    adata, 
    n_top_genes=2000, 
    flavor="seurat_v3", 
    layer="counts"
)

rsc.pp.pca(
    adata, 
    n_comps=50, 
    mask_var="highly_variable", 
    random_state=RANDOM_STATE
)

## 2. Build spatial graph

We build the spatial neighborhood graph by Delaunay triangulation with
`rsc.gr.spatial_neighbors_delaunay`. Triangulation on its own connects
cells across tissue gaps and around the convex hull, producing edges far
longer than any real cell-cell contact, so `percentile=99` prunes the
longest 1% of edges.

In [ ]:
%%time
rsc.gr.spatial_neighbors_delaunay(adata, percentile=99)

## 3. Cell type annotation

No reference atlas is wired up in this tutorial, so we use a
**marker-gene-set scoring** approach via `rsc.dcg.ulm`
(decoupler_gpu's univariate linear model.

Scores are averaged within each unsupervised leiden cluster and the label is assigned by `argmax` on the cluster means. Averaging over a cluster is far more robust than scoring each cell independently.

**Marker panel**

- **Tumor substates** reported for *this* dataset in Janesick et al 2026, AACR
  poster for the 10x introduction of the Atera technology (breast-cancer section).

In [ ]:
CELL_TYPE_MARKERS: dict[str, list[str]] = {
    # -- tumour epithelium (Janesick et al. AACR 2026 poster) --
    "Invasive_Tumor": ["CCND1", "MYEOV", "TPCN2", "LTO1", "SMIM38", "FGF19", "FGF4", "FADD", "CTTN"],  # 11q13, Fig. 9C
    "Invasive_Tumor_G1S": ["PCNA", "MCM2", "MCM5", "TYMS", "FEN1", "GINS2"],              # S-phase, Fig. 8
    "Invasive_Tumor_Mitotic": ["C21orf58", "ESPL1", "MELK", "NCAPG2", "TOP2A", "CENPF"],  # Fig. 8 + G2/M
    "DCIS_Basal_like": ["KIT", "PROM1", "GABRP", "SYNM", "CCL28"],                        # Fig. 7A
    "DCIS_Luminal_like": ["ANKRD30A", "ESR1", "NIBAN1", "BMPR1B", "SEC23B", "SLC24A3", "BARX2", "TSKU"],  # Fig. 7B
    "Apocrine": ["PIP", "AR", "HPGD", "ACSM1"],                                           # PIP per Fig. 6
    "Myoepithelial": ["TP63", "OXTR", "KRT14", "KRT17", "KRT5", "ACTA2", "MYLK"],         # epithelial + contractile
    # -- stroma / vasculature --
    "CAF_Neoplastic": ["PDGFRA", "DCN", "LUM", "COL1A2", "CFD", "C7"],          # pan-CAF + non-activated
    "CAF_Invasive": ["COL11A1", "COL10A1", "MMP11", "INHBA", "THBS2", "CTHRC1"], # invasion-associated stroma
    "CXCL14_pos": ["CXCL14"],                                                   # only marker given in Fig. 6
    "Pericyte": ["RGS5", "NOTCH3", "KCNJ8", "HIGD1B", "ABCC9"],                 # PDGFRB dropped (CAFs)
    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "EMCN", "PLVAP"],
    # -- immune --
    "T_Cell": ["CD3D", "CD3E", "CD3G", "CD2", "CD247", "CD6"],                  # CD4 / IL7R dropped
    "B_Cell": ["MS4A1", "CD19", "PAX5", "BANK1", "BLK", "FCRL1"],               # Fig. 4B genes; CD79A/B dropped
    "Plasma_Cell": ["MZB1", "JCHAIN", "TNFRSF17", "DERL3"],                     # non-Ig -> less spill-over
    "Macrophage": ["CD68", "CD163", "C1QA", "C1QB", "C1QC", "MS4A7"],
    "Dendritic_Cell": ["FLT3", "CLEC9A", "XCR1", "CD1C", "CLEC10A", "LAMP3"],
    # -- not labelled in Fig. 6: candidates for its 2 small unlabelled clusters --
    "Lymphatic_Endothelial": ["PROX1", "CCL21", "MMRN1", "FLT4", "LYVE1"],     # PDPN dropped
    "Mast_Cell": ["TPSAB1", "TPSB2", "CPA3", "MS4A2", "HDC"],                  # KIT left to DCIS_Basal_like
}

celltype_net = (
    pd.Series(CELL_TYPE_MARKERS).explode().rename_axis("source").reset_index(name="target")
)
celltype_net.groupby("source").size()

In [ ]:
adata.X = adata.layers["lognorm"]

rsc.dcg.ulm(adata, celltype_net, layer="lognorm", tmin=2)
adata.obsm["score_celltype"] = adata.obsm.pop("score_ulm")
adata.obsm.pop("padj_ulm", None)

score_ct = adata.obsm["score_celltype"]

In [ ]:
rsc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca", random_state=RANDOM_STATE)
rsc.tl.leiden(adata, resolution=1.0, key_added="leiden_expr", random_state=RANDOM_STATE)
rsc.tl.umap(adata, random_state=RANDOM_STATE)

In [ ]:
# Robust label: mean decoupler score per expression cluster
cluster_mean_score = score_ct.groupby(adata.obs["leiden_expr"].values).mean()
cluster_to_celltype = cluster_mean_score.idxmax(axis=1)

adata.obs["cell_type"] = (
    adata.obs["leiden_expr"].map(cluster_to_celltype).astype("category")
)

adata.obs["cell_type"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sc.pl.umap(adata, color="leiden_expr", ax=axes[0], show=False, legend_loc="on data", title="Leiden (expression)")
sc.pl.umap(adata, color="cell_type", ax=axes[1], show=False, title="Cell type (cluster-level argmax)")

In [ ]:
# Categorical spatial layout via squidpy
sq.pl.spatial_scatter(
    adata, color="cell_type", shape=None, img=False,
    palette="tab20", size=1.5, figsize=(7, 6),
)

## 4. Identifying spatial structure

### 4.1 Spatially aware clustering (spatial domains / niches)

`rsc.gr.calculate_niche_cellcharter` builds the CellCharter feature matrix 
concatenating expression with its mean over the 1-, 2- and 3-hop shells of 
the spatial graph, then PCA-reduces it and fits a Gaussian mixture.

- **Aggregation in PCA space, not raw expression.** With 18,028 genes and
  `distance=3`, the default feature matrix is 72,112 columns wide, and each
  shell product `Âₖ @ X` is a sparse × sparse multiply that fails on this
  dataset with `CUSPARSE_STATUS_INSUFFICIENT_RESOURCES`. We instead aggregate
  the dense 50-component `X_pca`, which makes every shell product a cheap
  sparse × dense matmul and matches the original CellCharter paper,
  which also aggregates in PCA space rather than over raw counts.

- **Choosing `k`.** `n_components` is fixed per run, we use k=9, fine enough to separate
  tumor, stromal and immune sub-niches while staying interpretable, and in the
  same range as the ~10 spatial clusters reported for this dataset in Janesick et al 2026, AACR
  neighborhood analysis.

In [ ]:
%%time 

niche_ad = ad.AnnData(
    X=rsc.get.X_to_CPU(adata.obsm["X_pca"]),
    obs=pd.DataFrame(index=adata.obs_names),
)

niche_ad.obsp["spatial_connectivities"] = adata.obsp["spatial_connectivities"]

rsc.gr.calculate_niche_cellcharter(niche_ad, n_components=9, random_state=RANDOM_STATE)

adata.obs["cellcharter_niche"] = niche_ad.obs["cellcharter_niche"].astype("category").values

adata.obs["cellcharter_niche"].value_counts()

In [ ]:
sq.pl.spatial_scatter(
    adata, 
    library_id='region',
    color="cellcharter_niche", 
    shape=None, 
    img=False,
    size=1 
)

### 4.2 Niche characterization

Characterize the spatial domains from 4.1 two ways:

- **Composition** — cell-type mixture per niche (from Section 3 labels).
- **Function** — mean PROGENy footprint score per niche, via
  `rsc.dcg.ulm`. PROGENy is retrieved from `OmniPath`,
  and its 14 pathways carry signed, fitted weights per gene, so a score
  reflects how far a cell's expression departs from the pathway footprint in
  both directions.

In [ ]:
composition = (
    pd.crosstab(adata.obs["cellcharter_niche"], adata.obs["cell_type"], normalize="index") * 100
)

fig, ax = plt.subplots(figsize=(10, 5))
composition.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("% of cells in niche")
ax.set_title(f"Cell-type composition per spatial domain")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
fig.tight_layout()

composition.round(1)

In [ ]:
# PROGENy pathway footprints, retrieved from OmniPath. Weights are signed, which is
# what ulm expects; the curated lists this replaces were effectively unweighted.

N_PROGENY_GENES = 500

progeny_net = (
    op.requests.Annotations.get(resources="PROGENy")
    .pivot(index=["record_id", "genesymbol"], columns="label", values="value")
    .reset_index()
    .rename(columns={"pathway": "source", "genesymbol": "target"})
    .astype({"weight": float, "p_value": float})
    .sort_values("p_value")
    .drop_duplicates(subset=["source", "target"])
    .groupby("source")
    .head(N_PROGENY_GENES)[["source", "target", "weight"]]
    .reset_index(drop=True)
)

rsc.dcg.ulm(adata, progeny_net, layer="lognorm")

adata.obsm["score_pathway"] = adata.obsm.pop("score_ulm")

adata.obsm.pop("padj_ulm", None)

score_pw = adata.obsm["score_pathway"]
niche_pathway = score_pw.groupby(adata.obs["cellcharter_niche"].values).mean()
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(niche_pathway, cmap="vlag", center=0, annot=True, fmt=".1f", ax=ax)
ax.set_title("Mean PROGENy pathway activity per spatial domain")
fig.tight_layout()

### 4.3 Spatial structure metrics: 

co-occurrence, neighborhood enrichment, ligand-receptor signaling

Three classic squidpy-style analyses, complementary to the niches
above (which summarize *local neighborhoods*; these instead
summarize *pairwise* spatial relationships between cell types):

- **`rsc.gr.co_occurrence`** — GPU. For each cell-type pair, how does
  the probability of finding the second type change with distance
  from the first? We highlight the hypoxic-tumor state against
  immune/stromal types.
- **`squidpy.gr.nhood_enrichment`** — CPU (no `rsc.gr` equivalent
  yet). Permutation test for whether cell-type pairs are spatial
  neighbors more/less often than chance, summarized as a z-score
  matrix.
- **`rsc.gr.ligrec`** — GPU. Squidpy-style permutation test for
  ligand–receptor pairs enriched between cluster pairs. We retrieved
  LR ineraction netwrok from `omnipath`– ~11k pairs, restricted to those
  whose partners are all on the panel — because on GPU the full screen
  costs seconds, filtering happens only at the plotting step: a dotplot of ~10k rows is
  unreadable, so we show the interactions significant across the most
  cluster pairs.

In [ ]:
%%time

rsc.gr.co_occurrence(
    adata,
    cluster_key="cell_type",
    interval=np.linspace(0, 250, 26),
    multi_gpu=False,
)
sq.pl.co_occurrence(
    adata,
    cluster_key="cell_type",
    clusters="Invasive_Tumor",
    figsize=(7, 5),
)

In [ ]:
rsc.gr.nhood_enrichment(adata, cluster_key="cell_type", n_perms=500, seed=RANDOM_STATE)
sq.pl.nhood_enrichment(adata, cluster_key="cell_type", figsize=(8, 7))

In [ ]:
interactions = rsc.squidpy_gpu._ligrec._get_interactions()

In [ ]:
%%time
lr_genes = (
    interactions[["source", "target"]]
    .stack()
    .astype(str)
    .str.split("_")
    .explode()
    .unique()
)
lr_genes = adata.var_names[adata.var_names.isin(lr_genes)]

adata_lr = adata[:, lr_genes].copy()
adata_lr.X = adata_lr.layers["lognorm"]
adata_lr.obs["cell_type"] = adata_lr.obs["cell_type"].cat.remove_unused_categories()
rsc.get.anndata_to_CPU(adata_lr, convert_all=True)  # ligrec's setup is host-side pandas; it moves the matrix to GPU itself

cp.random.seed(RANDOM_STATE)  # rsc.gr.ligrec takes no seed argument
ligrec_res = rsc.gr.ligrec(
    adata_lr,
    n_perms=1000,
    interactions=interactions,
    cluster_key="cell_type",
    copy=True,
    use_raw=False,
    corr_method="fdr_bh",
)
del adata_lr

In [ ]:
lr_clusters = [
    "Invasive_Tumor", "DCIS_Basal_like", "T_Cell",
    "Macrophage", "CAF_Neoplastic", "Endothelial",
]

ALPHA, TOP_N, MIN_SIG = 0.01, 10, 5

pv, mu = ligrec_res["pvalues"], ligrec_res["means"]

# Rank only over the cluster pairs that will actually be drawn.
cols = [c for c in pv.columns if c[0] in lr_clusters and c[1] in lr_clusters]

# Densify first: boolean reductions on sparse frames are not reliable across pandas versions.
pv_d, mu_d = pv[cols].sparse.to_dense(), mu[cols].sparse.to_dense()
sig = pv_d <= ALPHA
rank = pd.DataFrame({
    "n_sig": sig.sum(axis=1),
    "max_mean": mu_d.where(sig).max(axis=1),
})
# Keep interactions significant in at least MIN_SIG cluster pairs, then rank by
# their strongest significant mean expression.
top = (
    rank[rank["n_sig"] >= MIN_SIG]
    .sort_values("max_mean", ascending=False)
    .head(TOP_N)
    .index
)
ligrec_top = {k: df.loc[top] for k, df in ligrec_res.items()}

sq.pl.ligrec(
    ligrec_top,
    source_groups=lr_clusters,
    target_groups=lr_clusters,
    alpha=ALPHA,
    pvalue_threshold=ALPHA,
    title="Top ligand-receptor interactions across cell-type pairs",
    figsize=(0.7 * len(lr_clusters) + 4, 4),
)

## 5. Spatially variable genes & gene-level tasks

Spatially variable gene (SVG) detection via Moran's I,
`rsc.gr.spatial_autocorr(..., mode="moran")` — the GPU-accelerated
drop-in for `squidpy.gr.spatial_autocorr`.

In [ ]:
%%time
moran_df = rsc.gr.spatial_autocorr(adata, mode="moran", layer="lognorm", copy=True)
moran_df = moran_df.sort_values("I", ascending=False)
moran_df.head(10)

In [ ]:
%%time
top_svgs = moran_df.index[:6].tolist()

# squidpy plotting is CPU/matplotlib-based and can't read CuPy-backed
# layers directly, so pull this AnnData back to host once before plotting 

rsc.get.anndata_to_CPU(adata, convert_all=True)

titles = {g: f"{g}  (Moran's I={moran_df.loc[g, 'I']:.2f})" for g in top_svgs}
axes = sq.pl.spatial_scatter(
    adata, 
    library_id='region',
    color=top_svgs, 
    layer="lognorm", 
    shape=None, 
    img=False,
    cmap="viridis", 
    size=1.5, 
    ncols=3, 
    figsize=(15, 9), 
    return_ax=True,
)
for ax, gene in zip(np.atleast_1d(axes), top_svgs):
    ax.set_title(titles[gene])

`CCND1` is the most spatially variable gene (Moran's I = 0.73). Expression is highest in the large invasive nests (centre/right) and low in the in-situ ducts on the left and in stroma. This matches the 11q13 (CCND1) gain restricted to invasive cells in Janesick et al., AACR 2026. 

In [ ]:
print("Total Processing time: %s" % (time.time() - notebook_start))